# Modbus/TCP PCAP Feature Extraction for Anomaly Detection

This notebook extracts Modbus/TCP fields from PCAP files using tshark and exports them to CSV for ML-based anomaly detection.

In [56]:
# Requires tshark (Wireshark CLI) to be installed and in PATH
# Download from: https://www.wireshark.org/download.html

In [57]:
import subprocess
import json
import pandas as pd
from datetime import datetime
from pathlib import Path
import csv
import os

In [58]:
def extract_modbus_features(pcap_path: str) -> list[dict]:
    """
    Extract Modbus/TCP fields from a PCAP file using tshark.
    
    Args:
        pcap_path: Path to the PCAP file
        
    Returns:
        List of dictionaries containing extracted features
    """
    # Define fields to extract (mbtcp = Modbus/TCP header, modbus = PDU)
    fields = [
        "frame.number",
        "frame.time_epoch",
        "frame.len",
        "ip.src",
        "ip.dst",
        "tcp.srcport",
        "tcp.dstport",
        "tcp.len",
        "mbtcp.trans_id",
        "mbtcp.prot_id",
        "mbtcp.len",
        "mbtcp.unit_id",
        "modbus.func_code",
        "modbus.reference_num",
        "modbus.word_cnt",
        "modbus.bit_cnt",
        "modbus.byte_cnt",
        "modbus.exception_code",
    ]
    
    # Build tshark command
    cmd = [
        "tshark",
        "-r", pcap_path,
        "-Y", "modbus",
        "-T", "fields",
        "-E", "header=y",
        "-E", "separator=|",
        "-E", "quote=d",
    ]
    
    # Add field arguments
    for field in fields:
        cmd.extend(["-e", field])
    
    # Run tshark
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        raise RuntimeError(f"tshark failed: {result.stderr}")
    
    # Parse output
    lines = result.stdout.strip().split("\n")
    if len(lines) < 2:
        return []
    
    headers = lines[0].split("|")
    records = []
    
    def parse_int(val):
        """Parse int, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return int(val)
        except ValueError:
            return None
    
    def parse_float(val):
        """Parse float, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return float(val)
        except ValueError:
            return None
    
    def parse_str(val):
        """Parse string, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        return val or None
    
    for line in lines[1:]:
        values = line.split("|")
        row = dict(zip(headers, values))
        
        timestamp = parse_float(row.get('frame.time_epoch'))
        
        record = {
            'packet_number': parse_int(row.get('frame.number')),
            'timestamp': timestamp,
            'datetime': datetime.fromtimestamp(timestamp).isoformat() if timestamp else None,
            'src_ip': parse_str(row.get('ip.src')),
            'dst_ip': parse_str(row.get('ip.dst')),
            'src_port': parse_int(row.get('tcp.srcport')),
            'dst_port': parse_int(row.get('tcp.dstport')),
            'tcp_len': parse_int(row.get('tcp.len')),
            'transaction_id': parse_int(row.get('mbtcp.trans_id')),
            'protocol_id': parse_int(row.get('mbtcp.prot_id')),
            'modbus_length': parse_int(row.get('mbtcp.len')),
            'unit_id': parse_int(row.get('mbtcp.unit_id')),
            'function_code': parse_int(row.get('modbus.func_code')),
            'reference_num': parse_int(row.get('modbus.reference_num')),
            'word_count': parse_int(row.get('modbus.word_cnt')),
            'bit_count': parse_int(row.get('modbus.bit_cnt')),
            'byte_count': parse_int(row.get('modbus.byte_cnt')),
            'exception_code': parse_int(row.get('modbus.exception_code')),
            'pkt_len': parse_int(row.get('frame.len')),
        }
        
        # Derive additional features
        record['is_request'] = 1 if record['dst_port'] == 502 else 0
        record['is_response'] = 1 if record['src_port'] == 502 else 0
        record['is_exception'] = 1 if record['exception_code'] is not None else 0
        
        records.append(record)
    
    return records

In [59]:
# Modbus function code reference for labeling
MODBUS_FUNCTION_CODES = {
    1: 'Read Coils',
    2: 'Read Discrete Inputs',
    3: 'Read Holding Registers',
    4: 'Read Input Registers',
    5: 'Write Single Coil',
    6: 'Write Single Register',
    7: 'Read Exception Status',
    8: 'Diagnostics',
    15: 'Write Multiple Coils',
    16: 'Write Multiple Registers',
    22: 'Mask Write Register',
    23: 'Read/Write Multiple Registers',
    43: 'Read Device Identification',
}

def add_function_name(df: pd.DataFrame) -> pd.DataFrame:
    """Add human-readable function code names."""
    df['function_name'] = df['function_code'].map(MODBUS_FUNCTION_CODES).fillna('Unknown')
    return df

In [60]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add time-based features useful for anomaly detection.
    """
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # Inter-arrival time
    df['inter_arrival_time'] = df['timestamp'].diff()
    
    # Time since first packet
    df['time_from_start'] = df['timestamp'] - df['timestamp'].iloc[0]
    
    # Rolling statistics (last 10 packets)
    df['rolling_iat_mean'] = df['inter_arrival_time'].rolling(window=10, min_periods=1).mean()
    df['rolling_iat_std'] = df['inter_arrival_time'].rolling(window=10, min_periods=1).std()
    
    return df

In [61]:
def add_flow_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add flow-based features for each unique connection pair.
    """
    # Create flow identifier
    df['flow_id'] = df.apply(
        lambda x: f"{min(str(x['src_ip']), str(x['dst_ip']))}_{max(str(x['src_ip']), str(x['dst_ip']))}",
        axis=1
    )
    
    # Packet count per flow
    df['flow_pkt_count'] = df.groupby('flow_id').cumcount() + 1
    
    # Function code diversity per flow (rolling)
    df['unique_func_codes'] = df.groupby('flow_id')['function_code'].transform(
        lambda x: x.expanding().apply(lambda y: y.nunique())
    )
    
    return df

## Usage Example

In [ ]:
# === CONFIGURE YOUR PCAP FILE PATH HERE ===
relative_path = "data\\raw\\captures1_v2\\clean\\eth2dump-clean-1h_1.pcap"
absolute_path = (Path("..") / relative_path).resolve()

print(f"Using PCAP file at: {absolute_path}")

PCAP_PATH = absolute_path  # Update this path
OUTPUT_CSV = "modbus_features.csv"

Using PCAP file at: C:\Users\jorel\OneDrive\Documents\CODE STUFF\DOE\BNL\foundational_model_for_energy_security\data\raw\captures1_v2\clean\eth2dump-clean-1h_1.pcap


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'WindowsPath'

In [ ]:
# Extract raw Modbus features
print(f"Processing: {PCAP_PATH}")
records = extract_modbus_features(PCAP_PATH)
print(f"Extracted {len(records)} Modbus packets")

Processing: 


RuntimeError: tshark failed: tshark: The file "" doesn't exist.


In [ ]:
# Convert to DataFrame and add derived features
df = pd.DataFrame(records)

if len(df) > 0:
    df = add_function_name(df)
    df = add_temporal_features(df)
    df = add_flow_features(df)
    
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
else:
    print("No Modbus packets found in PCAP!")

In [ ]:
# Preview the data
df.head(10)

In [ ]:
# Summary statistics
print("Function Code Distribution:")
print(df['function_name'].value_counts())
print("\nBasic Statistics:")
df[['pkt_len', 'inter_arrival_time', 'modbus_length']].describe()

In [ ]:
# Export to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved to: {OUTPUT_CSV}")

## Feature Summary for Anomaly Detection

| Feature Category | Fields | Use Case |
|-----------------|--------|----------|
| **Protocol** | `function_code`, `unit_id`, `transaction_id` | Detect unauthorized commands |
| **Payload** | `reference_num`, `word_count`, `byte_count`, `reg_value` | Detect data manipulation |
| **Temporal** | `inter_arrival_time`, `rolling_iat_*` | Detect timing anomalies, DoS |
| **Flow** | `flow_pkt_count`, `unique_func_codes` | Detect reconnaissance, scanning |
| **Error** | `is_exception`, `exception_code` | Detect probing, fuzzing |